# Multi-model metric distributions at 300 tokens

Compare four models for perplexity, D-score, prompt mean, and prompt standard deviation. Each figure keeps the dataset rows and HG/MG columns from the reference distribution plots.

In [ ]:
from __future__ import annotations

import os
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.logprob_metrics_distributions_utils import (
    compute_prefix_metrics,
    iter_logprob_rows,
)

sns.set_theme(style="whitegrid", context="notebook")

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
PLOT_OUTPUT_ROOT = DATA_ROOT / "plots" / "multi_model_metric_distribution"
PLOT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Parameters

In [ ]:
TOKEN_LENGTH = 300
SELECTED_DATASETS = [
    "kaggle_train",
    "mage_train",
    "outfox_chatgpt_train",
    "bbc_news_content",
    "semeval2024_task8_subtask_c_train",
]
SELECTED_MODELS = [
    "gemma_3_1b_it",
    "gemma_3_270m_it",
    "littlelamb",
    "smollm2_360m",
]
METRICS = [
    "perplexity",
    "d_score_top20",
    "prompt_mu_score",
    "prompt_sigma_score",
]
MODEL_PALETTE = dict(zip(SELECTED_MODELS, sns.color_palette("viridis", len(SELECTED_MODELS))))
SPLIT_ORDER = ["HG", "MG"]
MAX_TEXTS_PER_DATASET_SPLIT = 100
SAMPLING_SEED = 42
GENERATE_LIMITED_OUTPUTS = True
GENERATE_ALL_TEXTS_OUTPUTS = True

In [ ]:
def load_metric_frame(max_texts_per_dataset_split):
    if max_texts_per_dataset_split is not None and max_texts_per_dataset_split <= 0:
        raise ValueError("MAX_TEXTS_PER_DATASET_SPLIT must be positive or None")

    selected_ids = None
    if max_texts_per_dataset_split is not None:
        ids_by_group_and_model = defaultdict(lambda: defaultdict(set))
        for row in iter_logprob_rows(
            PROCESSED_ROOT,
            selected_datasets=SELECTED_DATASETS,
            selected_models=SELECTED_MODELS,
        ):
            key = (row["dataset"], row["split"])
            if len(row.get("tokens", [])) >= TOKEN_LENGTH:
                ids_by_group_and_model[key][row["model"]].add(row["id"])

        rng = random.Random(SAMPLING_SEED)
        selected_ids = defaultdict(dict)
        for key, model_ids in ids_by_group_and_model.items():
            for model, eligible_ids in model_ids.items():
                ordered_ids = sorted(eligible_ids)
                selected_ids[key][model] = set(
                    rng.sample(ordered_ids, min(max_texts_per_dataset_split, len(ordered_ids)))
                )

    records = []
    for row in iter_logprob_rows(
        PROCESSED_ROOT,
        selected_datasets=SELECTED_DATASETS,
        selected_models=SELECTED_MODELS,
    ):
        key = (row["dataset"], row["split"])
        if (
            len(row.get("tokens", [])) < TOKEN_LENGTH
            or (selected_ids is not None and row["id"] not in selected_ids.get(key, {}).get(row["model"], set()))
        ):
            continue
        prefix = next(
            (item for item in compute_prefix_metrics(row, step=TOKEN_LENGTH)
             if item["token_length"] == TOKEN_LENGTH),
            None,
        )
        if prefix is not None:
            records.append(prefix | {"id": row["id"], "dataset": row["dataset"], "split": row["split"], "model": row["model"]})
    return pd.DataFrame(records)

output_modes = {}
if GENERATE_LIMITED_OUTPUTS:
    output_modes[f"capped_{MAX_TEXTS_PER_DATASET_SPLIT}"] = MAX_TEXTS_PER_DATASET_SPLIT
if GENERATE_ALL_TEXTS_OUTPUTS:
    output_modes["all_texts"] = None

plot_frames = {name: load_metric_frame(limit) for name, limit in output_modes.items()}
for mode_name, frame in plot_frames.items():
    frame["model"] = pd.Categorical(frame["model"], categories=SELECTED_MODELS, ordered=True)
    print(f"{mode_name}: loaded {len(frame):,} rows at token length {TOKEN_LENGTH}")
print(f"Models: {', '.join(SELECTED_MODELS)}")
print(f"Modes: {', '.join(output_modes)}")

## Distribution plots

In [ ]:
for mode_name, plot_df in plot_frames.items():
    datasets = sorted(plot_df["dataset"].dropna().unique())
    for metric in METRICS:
        metric_df = plot_df.dropna(subset=[metric]).copy()
        metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")
        metric_df = metric_df.dropna(subset=[metric])
        if metric_df.empty:
            print(f"Skipping {mode_name}/{metric}: no values at token length {TOKEN_LENGTH}")
            continue

        grid = sns.FacetGrid(
            metric_df, row="dataset", col="split", hue="model",
            row_order=datasets, col_order=SPLIT_ORDER, hue_order=SELECTED_MODELS,
            palette=MODEL_PALETTE, height=1.65, aspect=2.4,
            margin_titles=False, sharex=True, sharey=False,
        )
        grid.map_dataframe(sns.kdeplot, x=metric, fill=False, linewidth=1.8, warn_singular=False)
        grid.set_titles("")
        grid.figure.subplots_adjust(hspace=0.55, top=0.86)
        grid.figure.canvas.draw()
        for row_index, dataset in enumerate(datasets):
            for col_index, split in enumerate(SPLIT_ORDER):
                grid.axes[row_index, col_index].set_title(split if row_index == 0 else "", pad=8)
            left_axis = grid.axes[row_index, 0].get_position()
            right_axis = grid.axes[row_index, -1].get_position()
            grid.figure.text((left_axis.x0 + right_axis.x1) / 2, left_axis.y1 + 0.035, dataset, ha="center", va="bottom", fontsize=10)
        if metric == "perplexity":
            grid.set(xscale="log")
        grid.set_axis_labels(metric, "Density")
        grid.add_legend(title="Model")
        grid.figure.subplots_adjust(top=0.86)
        grid.figure.suptitle(f"{metric}: {mode_name} distribution at {TOKEN_LENGTH} tokens", fontsize=15, fontweight="bold")
        output_path = PLOT_OUTPUT_ROOT / f"ridgeline_{metric}__all_models__{TOKEN_LENGTH}_tokens__{mode_name}.png"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grid.figure.savefig(str(output_path), dpi=300, bbox_inches="tight")
        plt.show()

## Distribution summary tables

Each table reports the distribution mean and standard deviation in separate columns for every dataset, LLM, and HG/MG split combination.

In [ ]:
from IPython.display import display

for mode_name, plot_df in plot_frames.items():
    datasets = sorted(plot_df["dataset"].dropna().unique())
    for metric in METRICS:
        metric_df = plot_df.copy()
        metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")
        metric_df = metric_df.dropna(subset=[metric])
        summary = (
            metric_df.groupby(["dataset", "model", "split"], observed=True)[metric]
            .agg(mean="mean", std="std")
            .reindex(pd.MultiIndex.from_product([datasets, SELECTED_MODELS, SPLIT_ORDER], names=["dataset", "model", "split"]))
        )
        table = summary.unstack(["model", "split"])
        table.columns = table.columns.set_names(["Statistic", "LLM", "Split"])
        table = table.reorder_levels(["LLM", "Split", "Statistic"], axis=1).sort_index(axis=1, level=["LLM", "Split"])
        table.index.name = "Dataset"
        table.to_csv(PLOT_OUTPUT_ROOT / f"summary_{metric}__{TOKEN_LENGTH}_tokens__{mode_name}.csv")
        display(table.style.format("{:.4g}").set_caption(f"{metric}: {mode_name} mean and standard deviation at {TOKEN_LENGTH} tokens"))